In [ ]:
from pathlib import Path
import pandas as pd


# ============================================================
# Configuration
# ============================================================

BASE = Path("../dataset")

FILES = {
    "carbon": BASE / "SustainableFoodDB_CarbonFootprints.csv",
    "flavor_entities": BASE / "FlavorDB_entities.csv",
    "flavor_molecules": BASE / "FlavorDB_molecules.csv",
    "flavor_links": BASE / "FlavorDB_entity_molecules.csv",
    "recipes": BASE / "RecipeDB_full" / "recipes.csv",
    "recipe_ingredients": BASE / "RecipeDB_full" / "recipe_ingredients.csv",
    "recipe_nutrition": BASE / "RecipeDB_full" / "recipe_nutrition.csv",
    "recipe_processes": BASE / "RecipeDB_full" / "recipe_processes.csv",
    "recipe_utensils": BASE / "RecipeDB_full" / "recipe_utensils.csv",
    "recipe_steps": BASE / "RecipeDB_full" / "recipe_steps.csv",
    "recipe_links": BASE / "RecipeDB_full" / "recipe_links.csv",
}
# ============================================================
# Helper functions
# ============================================================

def print_section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def audit_dataframe(name, df):
    print_section(f"{name.upper()}")

    print(f"Rows              : {len(df):,}")
    print(f"Columns           : {len(df.columns):,}")
    print(f"Duplicate rows    : {df.duplicated().sum():,}")

    print("\nColumns:")
    for column in df.columns:
        print(f"  - {column}")

    print("\nData types:")
    print(df.dtypes.to_string())

    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(2)

    audit = pd.DataFrame({
        "missing": missing,
        "missing_pct": missing_pct,
        "unique_values": df.nunique(dropna=True)
    })

    print("\nMissing / Unique value audit:")
    print(audit.to_string())

    print("\nFirst 3 rows:")
    print(df.head(3).to_string(index=False))


def check_id_column(df, column):
    if column not in df.columns:
        return

    total = len(df)
    unique = df[column].nunique(dropna=True)
    missing = df[column].isna().sum()
    duplicates = total - unique - missing

    print(f"\nID column: {column}")
    print(f"  Total values     : {total:,}")
    print(f"  Unique values    : {unique:,}")
    print(f"  Missing values   : {missing:,}")
    print(f"  Duplicate values : {duplicates:,}")


# ============================================================
# Load datasets
# ============================================================

print_section("DATASET AVAILABILITY")

datasets = {}

for name, path in FILES.items():
    if path.exists():
        try:
            df = pd.read_csv(path)
            datasets[name] = df

            print(
                f"[OK]   {name:<22} "
                f"{len(df):>10,} rows x {len(df.columns):>3} columns"
            )

        except Exception as e:
            print(f"[ERROR] {name:<22} Could not read file")
            print(f"        {e}")

    else:
        print(f"[MISS] {name:<22} {path}")


# ============================================================
# Basic audit
# ============================================================

for name, df in datasets.items():
    audit_dataframe(name, df)


# ============================================================
# ID / key checks
# ============================================================

print_section("KEY / ID CHECKS")

id_columns = {
    "carbon": ["_id"],
    "flavor_entities": ["entity_id"],
    "flavor_molecules": ["pubchem_id"],
    "flavor_links": ["entity_id", "pubchem_id"],
    "recipes": ["recipe_id"],
    "recipe_ingredients": ["recipe_id"],
    "recipe_nutrition": ["recipe_id"],
    "recipe_processes": ["recipe_id"],
    "recipe_utensils": ["recipe_id"],
    "recipe_steps": ["recipe_id"],
    "recipe_links": ["recipe_id"],
}

for dataset_name, columns in id_columns.items():

    if dataset_name not in datasets:
        continue

    df = datasets[dataset_name]

    print(f"\n--- {dataset_name} ---")

    for column in columns:
        check_id_column(df, column)


# ============================================================
# Duplicate key checks
# ============================================================

print_section("DUPLICATE KEY CHECKS")

duplicate_checks = {
    "carbon": ["_id"],
    "flavor_entities": ["entity_id"],
    "flavor_molecules": ["pubchem_id"],
    "flavor_links": ["entity_id", "pubchem_id"],
    "recipes": ["recipe_id"],
}

for dataset_name, columns in duplicate_checks.items():

    if dataset_name not in datasets:
        continue

    df = datasets[dataset_name]

    print(f"\n--- {dataset_name} ---")

    if all(column in df.columns for column in columns):
        duplicate_count = df.duplicated(subset=columns).sum()

        print(
            f"Duplicate rows based on "
            f"{columns}: {duplicate_count:,}"
        )


# ============================================================
# Numeric column summary
# ============================================================

print_section("NUMERIC COLUMN SUMMARY")

for name, df in datasets.items():

    numeric_columns = df.select_dtypes(
        include="number"
    ).columns

    if len(numeric_columns) == 0:
        continue

    print(f"\n--- {name} ---")

    print(
        df[numeric_columns]
        .describe()
        .transpose()
        .round(3)
        .to_string()
    )


# ============================================================
# Categorical column summary
# ============================================================

print_section("CATEGORICAL COLUMN SUMMARY")

for name, df in datasets.items():

    categorical_columns = df.select_dtypes(
        include=["object", "category"]
    ).columns

    if len(categorical_columns) == 0:
        continue

    print(f"\n--- {name} ---")

    for column in categorical_columns:

        unique_count = df[column].nunique(dropna=True)

        print(
            f"\n{column}: "
            f"{unique_count:,} unique values"
        )

        if unique_count <= 20:
            print(
                df[column]
                .value_counts(dropna=False)
                .to_string()
            )


# ============================================================
# Dataset overview table
# ============================================================

print_section("FINAL DATASET OVERVIEW")

overview = []

for name, df in datasets.items():

    overview.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "total_missing_values": int(df.isna().sum().sum()),
        "memory_mb": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2),
            2
        )
    })

overview_df = pd.DataFrame(overview)

print(
    overview_df
    .sort_values("dataset")
    .to_string(index=False)
)


print_section("EDA 01 COMPLETE")

print("Basic dataset audit completed.")
print("Next step: dataset-specific EDA.")

NameError: name '__file__' is not defined